In [66]:
pip install praw pytrends pandas nltk emoji textblob

In [67]:
import warnings
warnings.filterwarnings("ignore")
import praw
import pandas as pd
import re
import string
import emoji
import nltk
from nltk.corpus import stopwords
from pytrends.request import TrendReq
from textblob import TextBlob

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [68]:
reddit = praw.Reddit(
    client_id="ANAhS6ga7S-dXxS7mxsn7w",
    client_secret="Dwowr2WXUgqGDJkHbOI2Sx1Jws7DfA",
    user_agent="social_analytics_project/1.0"
)

In [69]:
pytrends = TrendReq()
STOPWORDS = set(stopwords.words('english'))

keywords = ["politics", "nuclear bomb", "government", "war", "economy"]

pytrends.build_payload(keywords, timeframe='now 1-d')

In [70]:
trending = pytrends.related_queries()
import pandas as pd

trending = pytrends.related_queries()

dfs = []

for keyword in trending:
    df = trending[keyword]["top"]

    if df is not None:
        df["keyword"] = keyword
        dfs.append(df)

final_df = pd.concat(dfs, ignore_index=True)

final_df

,query,value,keyword
0,politics news,100,politics
1,what is politics,66,politics
2,us politics,51,politics
3,reddit politics,35,politics
4,politics meaning,34,politics
...,...,...,...
120,economy meaning,22,economy
121,circular economy,21,economy
122,dubai economy,19,economy
123,political economy,19,economy


In [71]:
search_terms = []
for k in trending:
    if trending[k]["top"] is not None:
        search_terms.extend(trending[k]["top"]["query"].head(3).tolist())

In [72]:
import logging
logging.getLogger("praw").setLevel(logging.ERROR)

posts = []

for term in search_terms:
    for submission in reddit.subreddit("politics+worldnews+news").search(term, limit=50):
        posts.append(submission.title + " " + submission.selftext)

df = pd.DataFrame(posts, columns=["text"])
df

,text
0,“CBS Evening News” Has Lost Over a Million Vie...
1,Elon Musk's social media platform X sues Minne...
2,Liz Truss resigns as prime minister | Politics...
3,‘If she dares’: China warns U.S. Official agai...
4,Opinion | GOP Calling Trump Coup Effort 'Legit...
...,...
727,Megathread: National Security Adviser John Bol...
728,Trump to Sign Spending Deal and Declare Nation...
729,Megathread: Treasury denies Democrats’ request...
730,President Trump Declares National Emergency at...


In [73]:
def remove_urls(text):
    return re.sub(r'https?://\S+|www\.\S+', '', text)

def remove_html(text):
    return re.sub(r'<.*?>', '', text)

def remove_numbers(text):
    return re.sub(r'\d+', '', text)

def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

def remove_stopwords(text):
    return " ".join([word for word in text.split() if word not in STOPWORDS])

def preprocess(text):
    text = emoji.demojize(text)
    text = remove_urls(text)
    text = remove_html(text)
    text = remove_numbers(text)
    text = remove_hashtags(text)
    text = remove_mentions(text)
    text = remove_punctuation(text)
    text = text.lower()
    text = remove_stopwords(text)
    return text


In [74]:
df["clean_text"] = df["text"].apply(preprocess)

In [75]:
df

,text,clean_text
0,“CBS Evening News” Has Lost Over a Million Vie...,“cbs evening news” lost million viewers since ...
1,Elon Musk's social media platform X sues Minne...,elon musks social media platform x sues minnes...
2,Liz Truss resigns as prime minister | Politics...,liz truss resigns prime minister politics news
3,‘If she dares’: China warns U.S. Official agai...,‘if dares’ china warns us official visiting ta...
4,Opinion | GOP Calling Trump Coup Effort 'Legit...,opinion gop calling trump coup effort legitima...
...,...,...
727,Megathread: National Security Adviser John Bol...,megathread national security adviser john bolt...
728,Trump to Sign Spending Deal and Declare Nation...,trump sign spending deal declare national emer...
729,Megathread: Treasury denies Democrats’ request...,megathread treasury denies democrats’ request ...
730,President Trump Declares National Emergency at...,president trump declares national emergency so...


In [76]:
def sentiment(text):
    score = TextBlob(text).sentiment.polarity
    if score > 0:
        return "positive"
    elif score < 0:
        return "negative"
    else:
        return "neutral"

df["sentiment"] = df["clean_text"].apply(sentiment)

result = df["sentiment"].value_counts()

print(result)

sentiment
neutral     279
positive    269
negative    184
Name: count, dtype: int64
